# Contrastive fine-tuning of BGE-M3 on weak SDG pairs

This Kaggle notebook fine-tunes the `BAAI/bge-m3` encoder with a contrastive objective on the weakest SDG pairs from SDGX, then trains a classifier head on SDGi and evaluates both SDGi and SDGX performance.

Requirements:
- GPU accelerator (T4/other CUDA)
- Internet enabled (to download `UNDP/sdgi-corpus` and `BAAI/bge-m3`)
- A Kaggle dataset containing `sdgx_clean.jsonl` (e.g. `mahnoorzamirr/sdg-clean`), mounted at `/kaggle/input/...`.

In [ ]:
%%time
"""Setup: install dependencies."""
!pip install -q sentence-transformers datasets scikit-learn torch

In [ ]:
%%time
import os
import json
import time
from collections import defaultdict
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

assert torch.cuda.is_available(), "This notebook requires a GPU runtime. Please enable GPU in Kaggle settings."
device = torch.device("cuda")
print("Using device:", device)

## SDG definitions

We define short, one-line SDG descriptions to use as contrastive targets.

In [ ]:
SDG_DEFINITIONS: Dict[int, str] = {
    1: "End poverty in all its forms everywhere.",
    2: "End hunger, achieve food security and improved nutrition, and promote sustainable agriculture.",
    3: "Ensure healthy lives and promote well-being for all at all ages.",
    4: "Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all.",
    5: "Achieve gender equality and empower all women and girls.",
    6: "Ensure availability and sustainable management of water and sanitation for all.",
    7: "Ensure access to affordable, reliable, sustainable and modern energy for all.",
    8: "Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all.",
    9: "Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation.",
    10: "Reduce inequality within and among countries.",
    11: "Make cities and human settlements inclusive, safe, resilient and sustainable.",
    12: "Ensure sustainable consumption and production patterns.",
    13: "Take urgent action to combat climate change and its impacts.",
    14: "Conserve and sustainably use the oceans, seas and marine resources for sustainable development.",
    15: "Protect, restore and promote sustainable use of terrestrial ecosystems and halt biodiversity loss.",
    16: "Promote peaceful and inclusive societies, provide access to justice for all and build effective, accountable institutions.",
    17: "Strengthen the means of implementation and revitalize the global partnership for sustainable development.",
}

SDG_WEAK_PAIRS = ["10_16", "10_11", "4_10"]  # from earlier SDGX analysis
print("Weak pairs:", SDG_WEAK_PAIRS)

## Load SDGi and SDGX

- SDGi comes from HuggingFace (`UNDP/sdgi-corpus`).
- SDGX comes from a Kaggle dataset with `sdgx_clean.jsonl`.

Adjust `SDGX_PATH` below to match your dataset mount path if needed.

In [ ]:
%%time
# Paths
SDGX_PATH = Path("/kaggle/input/sdg-clean/sdgx_clean.jsonl")  # adjust if different
assert SDGX_PATH.exists(), f"SDGX file not found at {SDGX_PATH}"

ds_sdgi = load_dataset("UNDP/sdgi-corpus")
train_sdgi = ds_sdgi["train"]
test_sdgi = ds_sdgi["test"]
print("SDGi train size:", len(train_sdgi))
print("SDGi test size:", len(test_sdgi))

sdgx_texts: List[str] = []
sdgx_labels: List[np.ndarray] = []
sdgx_types: List[str] = []
sdgx_pairs: List[str] = []

with SDGX_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        ex = json.loads(line)
        text = ex.get("text")
        if not text:
            continue
        y_vec = np.zeros(17, dtype=int)
        ex_type = ex.get("type")
        pair = str(ex.get("pair") or "")
        if ex_type == "easy":
            primary = ex.get("primary_sdg")
            if primary is not None and 1 <= int(primary) <= 17:
                y_vec[int(primary) - 1] = 1
        elif ex_type == "hard":
            for s in ex.get("sdgs") or []:
                try:
                    s_int = int(s)
                except Exception:
                    continue
                if 1 <= s_int <= 17:
                    y_vec[s_int - 1] = 1
        if y_vec.sum() == 0:
            continue
        sdgx_texts.append(text)
        sdgx_labels.append(y_vec)
        sdgx_types.append(str(ex_type))
        sdgx_pairs.append(pair)

sdgx_labels = np.stack(sdgx_labels, axis=0)
print("SDGX size:", len(sdgx_texts))
print("Hard examples in weak pairs:", sum(t == "hard" and p in SDG_WEAK_PAIRS for t, p in zip(sdgx_types, sdgx_pairs)))

## Build contrastive training pairs

We construct `InputExample(texts=[anchor, positive, hard_negative])` for SDGX weak pairs, and add in-batch SDGi `(text, definition)` pairs to prevent catastrophic forgetting.

In [ ]:
%%time
train_examples: List[InputExample] = []

# SDGX weak hard pairs
for text, t, pair, y in zip(sdgx_texts, sdgx_types, sdgx_pairs, sdgx_labels):
    if t != "hard" or pair not in SDG_WEAK_PAIRS:
        continue
    try:
        a_str, b_str = pair.split("_")
        a, b = int(a_str), int(b_str)
    except Exception:
        continue
    pos_a = SDG_DEFINITIONS[a]
    pos_b = SDG_DEFINITIONS[b]
    # choose a hard negative: pick an SDG that is not a or b
    neg_candidates = [k for k in SDG_DEFINITIONS.keys() if k not in (a, b)]
    if not neg_candidates:
        continue
    neg_sdg = neg_candidates[0]
    neg_def = SDG_DEFINITIONS[neg_sdg]
    # Two InputExamples per hard example
    train_examples.append(InputExample(texts=[text, pos_a, neg_def]))
    train_examples.append(InputExample(texts=[text, pos_b, neg_def]))

print("Contrastive examples from weak SDGX pairs:", len(train_examples))

# Add in-batch SDGi pairs: up to 50 examples per SDG from train
per_sdg_texts: Dict[int, List[str]] = {i: [] for i in range(1, 18)}
for ex in train_sdgi:
    ys = ex.get("labels") or []
    for sdg in ys:
        if 1 <= sdg <= 17 and len(per_sdg_texts[sdg]) < 50:
            per_sdg_texts[sdg].append(ex["text"])

for sdg, texts in per_sdg_texts.items():
    definition = SDG_DEFINITIONS[sdg]
    for text in texts:
        # For in-batch negatives, we only need anchor & positive;
        # extra negatives will be provided by other samples in the batch.
        train_examples.append(InputExample(texts=[text, definition]))

print("Total contrastive examples (with SDGi):", len(train_examples))

## Contrastive fine-tuning of BGE-M3

We use `MultipleNegativesRankingLoss` on the constructed pairs, training for 3 epochs.

In [ ]:
%%time
model_name = "BAAI/bge-m3"
contrastive_model = SentenceTransformer(model_name, device=device)
contrastive_model.max_seq_length = 512

train_dataloader = DataLoader(train_examples, batch_size=32, shuffle=True)
train_loss = losses.MultipleNegativesRankingLoss(contrastive_model)

num_epochs = 3
warmup_steps = 100

output_dir = Path("bge_m3_contrastive")
output_dir.mkdir(parents=True, exist_ok=True)

contrastive_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
)

contrastive_model.save(str(output_dir))
print("Saved fine-tuned encoder to", output_dir)

## Embed SDGi with fine-tuned encoder and train FFN head

We now embed SDGi train/test with the fine-tuned encoder and train a fresh FFN head with the same hyperparameters as the baseline (BCELoss, Adam lr=1e-3, epochs=30, batch_size=64).

In [ ]:
%%time
def sdgi_to_xy(split):
    texts = []
    labels = []
    for ex in split:
        texts.append(ex["text"])
        ys = ex.get("labels") or []
        y_vec = np.zeros(17, dtype=int)
        for sdg in ys:
            if 1 <= sdg <= 17:
                y_vec[sdg - 1] = 1
        labels.append(y_vec)
    return texts, np.stack(labels, axis=0)

train_texts, y_train = sdgi_to_xy(train_sdgi)
test_texts, y_test = sdgi_to_xy(test_sdgi)

print("Embedding SDGi train/test with contrastive encoder...")
X_train_emb = contrastive_model.encode(train_texts, batch_size=32, show_progress_bar=True)
X_test_emb = contrastive_model.encode(test_texts, batch_size=32, show_progress_bar=True)

X_train = torch.from_numpy(X_train_emb).float().to(device)
X_test = torch.from_numpy(X_test_emb).float().to(device)
y_train_t = torch.from_numpy(y_train).float().to(device)
y_test_t = torch.from_numpy(y_test).float().to(device)

train_ds = torch.utils.data.TensorDataset(X_train, y_train_t)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)

class FFN(nn.Module):
    def __init__(self, input_dim: int = 1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 17),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

ffn = FFN(input_dim=X_train_emb.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(ffn.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs_ffn = 30
for epoch in range(1, num_epochs_ffn + 1):
    epoch_loss = 0.0
    ffn.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = ffn(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_ds)
    print(f"Epoch {epoch}/{num_epochs_ffn} - loss: {epoch_loss:.4f}")

## Metrics on SDGi

We compute overall Micro/Macro F1, per-label F1, multi-label F1, and specifically inspect the weak pairs (10_16, 10_11, 4_10).

In [ ]:
%%time
ffn.eval()
with torch.no_grad():
    y_pred_scores = ffn(X_test).cpu().numpy()
y_pred = (y_pred_scores >= 0.5).astype(int)

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, labels=None) -> Dict:
    if labels is None:
        labels = list(range(17))
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    per_label_f1: Dict[int, float] = {}
    per_label_p: Dict[int, float] = {}
    per_label_r: Dict[int, float] = {}
    for idx in labels:
        sdg_id = idx + 1
        y_t = y_true[:, idx]
        y_p = y_pred[:, idx]
        per_label_f1[sdg_id] = float(f1_score(y_t, y_p, zero_division=0))
        per_label_p[sdg_id] = float(precision_score(y_t, y_p, zero_division=0))
        per_label_r[sdg_id] = float(recall_score(y_t, y_p, zero_division=0))
    micro_f1 = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    return {
        "per_label_f1": per_label_f1,
        "per_label_precision": per_label_p,
        "per_label_recall": per_label_r,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }

metrics_sdgi = compute_metrics(y_test, y_pred)
print("Overall SDGi micro F1:", metrics_sdgi["micro_f1"])
print("Overall SDGi macro F1:", metrics_sdgi["macro_f1"])

print("\nPer-SDG F1:")
for sdg in range(1, 18):
    f1 = metrics_sdgi["per_label_f1"][sdg]
    print(f"  SDG{sdg}: {f1:.3f}")

# Multi-label subset
mask_multi = y_test.sum(axis=1) > 1
if mask_multi.any():
    m_multi = compute_metrics(y_test[mask_multi], y_pred[mask_multi])
    print("\nMulti-label only: micro F1 =", m_multi["micro_f1"], ", macro F1 =", m_multi["macro_f1"])

# Weak pairs: focus on SDGi examples containing each pair
def pair_metrics(pair: str) -> Dict[str, float]:
    a_str, b_str = pair.split("_")
    a, b = int(a_str), int(b_str)
    idx_a, idx_b = a - 1, b - 1
    mask = (y_test[:, idx_a] == 1) & (y_test[:, idx_b] == 1)
    if not mask.any():
        return {"micro_f1": 0.0, "macro_f1": 0.0}
    m = compute_metrics(y_test[mask][:, [idx_a, idx_b]], y_pred[mask][:, [idx_a, idx_b]])
    return {"micro_f1": m["micro_f1"], "macro_f1": m["macro_f1"]}

pair_results_sdgi = {pair: pair_metrics(pair) for pair in SDG_WEAK_PAIRS}
print("\nSDGi weak pair metrics:")
for pair, m in pair_results_sdgi.items():
    print(f"  Pair {pair}: micro F1 = {m['micro_f1']:.3f}, macro F1 = {m['macro_f1']:.3f}")

## SDGX hard pair evaluation with contrastive encoder

We now reuse the trained FFN head on SDGX embeddings from the contrastive encoder, mirroring the earlier SDGX evaluation.

In [ ]:
%%time
print("Embedding SDGX hard examples with contrastive encoder...")
hard_mask = np.array(sdgx_types) == "hard"
texts_hard = [t for t, m in zip(sdgx_texts, hard_mask) if m]
labels_hard = sdgx_labels[hard_mask]
pairs_hard = np.array(sdgx_pairs)[hard_mask]

X_hard_emb = contrastive_model.encode(texts_hard, batch_size=32, show_progress_bar=True)
X_hard = torch.from_numpy(X_hard_emb).float().to(device)

ffn.eval()
with torch.no_grad():
    y_pred_hard_scores = ffn(X_hard).cpu().numpy()
y_pred_hard = (y_pred_hard_scores >= 0.5).astype(int)

metrics_hard_all = compute_metrics(labels_hard, y_pred_hard)
print("Overall SDGX hard-only micro F1:", metrics_hard_all["micro_f1"])
print("Overall SDGX hard-only macro F1:", metrics_hard_all["macro_f1"])

def per_pair_metrics_hard(pair: str) -> Dict[str, float]:
    """Macro F1 is averaged over only the two SDGs in this pair, not all 17."""
    mask = pairs_hard == pair
    if not mask.any():
        return {"micro_f1": 0.0, "macro_f1": 0.0}
    a_str, b_str = str(pair).split("_")
    ia, ib = int(a_str) - 1, int(b_str) - 1
    y_t = labels_hard[mask][:, [ia, ib]]
    y_p = y_pred_hard[mask][:, [ia, ib]]
    m = compute_metrics(y_t, y_p, labels=[0, 1])
    return {"micro_f1": m["micro_f1"], "macro_f1": m["macro_f1"]}

all_pairs = sorted(set(str(p) for p in pairs_hard if p))
pair_results_sdgx = {p: per_pair_metrics_hard(p) for p in all_pairs}

print("\nSDGX hard per-pair metrics (macro F1 = avg over the two SDGs in pair only):")
print(f"{'Pair':<8} {'Micro F1':>10} {'Macro F1':>10}")
print("-" * 30)
for pair in all_pairs:
    m = pair_results_sdgx[pair]
    print(f"{pair:<8} {m['micro_f1']:>10.3f} {m['macro_f1']:>10.3f}")

## Save metrics and model

We now save all relevant metrics to `contrastive_metrics.json` and keep the fine-tuned encoder + FFN head for download.

In [ ]:
%%time
metrics_all = {
    "sdgi": metrics_sdgi,
    "sdgi_weak_pairs": pair_results_sdgi,
    "sdgx_hard_overall": {
        "micro_f1": metrics_hard_all["micro_f1"],
        "macro_f1": metrics_hard_all["macro_f1"],
    },
    "sdgx_hard_pairs": pair_results_sdgx,
}

with open("contrastive_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_all, f, ensure_ascii=False, indent=2)

# Save FFN head
torch.save(ffn.state_dict(), "ffn_contrastive_sdgi.pt")

print("Saved metrics to contrastive_metrics.json")
print("Saved FFN head to ffn_contrastive_sdgi.pt")